# Set up

In [1]:
import sys
if sys.platform == 'linux':
    sys.path.append("/home/qix/MultiNeuronGLM")
else:
    sys.path.append("D:/Github/MultiNeuronGLM")

In [57]:
import pandas as pd
import utility_functions as utils
import GLM
from DataLoader import Allen_dataset
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
sns.set_theme()

In [62]:
# Load LFP data
start_time = 0.0
end_time = 0.50
padding = 0.1
V1 = Allen_dataset(fps=1000,
               start_time=start_time, 
               end_time=end_time,
               padding=padding,
#                    orientation=[0],
               session_id=757216464,
               selected_probes=['probeA', 'probeB', 'probeC', 'probeD', 'probeE', 'probeF'],
#                    temporal_frequency=[1,2,4],
               stimulus_condition_id=[275, 277, 246, 255, 272, 248, 283, 266, 274, 276, 286, 271, 268, 270],
#                stimulus_condition_id = [246, 247, 248, 249, 250, 251, 252, 253, 255, 256, 257, 258,
#                                        259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 271,
#                                        272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284,
#                                        285, 286, 270],
               stimulus_name='drifting_gratings')

# V1.get_lfp()
# V1.remove_padding(padding)
V1.get_trial_metric_per_unit_per_trial()
V1.get_trial_metric_per_unit_per_trial(metric_type='spike_times')
V1.get_running(method="mine")

/home/qix/anaconda3/lib/python3.9/site-packages/allensdk/brain_observatory/ecephys/stimulus_table/naming_utilities.py:154: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  movie_rows = table[stim_colname].str.contains(movie_re, na=False)
/home/qix/anaconda3/lib/python3.9/site-packages/allensdk/brain_observatory/ecephys/ecephys_session.py:1315: UserWarning: Session includes invalid time intervals that could be accessed with the attribute 'invalid_times',Spikes within these intervals are invalid and may need to be excluded from the analysis.
  warnings.warn("Session includes invalid time intervals that could be accessed with the attribute 'invalid_times',"


In [63]:
# Load selected group_id
import pickle
with open('group_id_selected_a_c/membership.pickle', 'rb') as handle:
    membership = pickle.load(handle)
with open('group_id_selected_a_c/condition_ids.pickle', 'rb') as handle:
    condition_ids = pickle.load(handle)

In [104]:
select_trials = V1.running_trial_index
num_basis_baseline = 10

model = GLM.PP_GLM(dataset=V1, 
                   select_trials=select_trials, 
                   membership=membership, 
                   condition_ids=condition_ids)
model.add_effect('inhomogeneous_baseline', num=num_basis_baseline)
model.add_effect('coupling', 'probeD', peaks_max=150, num=5, nonlinear=0.2 )
model.add_effect('coupling', 'probeE', peaks_max=150, num=5, nonlinear=0.2 )
model.fit('probeC', verbose=True)

Negative log likelihood is: 28071.10
aic/2 is: 28092.10


In [106]:
import statsmodels.api as sm

In [262]:
beta = sm.GLM(model.response, model.predictors, family=sm.families.Poisson()).fit().params
beta

array([-1.72262773e+00, -4.25714414e-01,  1.06328413e+00, -1.80501034e-01,
       -2.70633965e-01,  1.24366379e-01, -4.41164800e-01, -4.40278255e-01,
       -2.80232895e-01, -4.26767021e-01, -6.04913353e-01,  2.74496749e-02,
        1.87430258e-02,  1.70411002e-02, -2.33954263e-02,  2.43029193e-02,
        7.76782496e-02, -2.84670132e-02,  1.40034770e-03,  2.57664605e-03,
       -1.37590198e-03])

In [263]:
model.log_lmbd.flatten('F')

array([-2.14834214, -2.08430225, -2.021773  , ..., -1.59992223,
       -1.61521368, -1.59345898])

In [265]:
model.nll + + L2_pen * np.linalg.norm(beta*penalty_vec)**2

28072.182036910297

21

In [274]:
L2_pen = 1e-2
penalty_vec = np.ones((model.predictors.shape[1], 1))
penalty_vec[0] = 0
log_lmbda_hat, beta = GLM.poisson_regression(model.predictors, model.response, L2_pen = L2_pen)

(21, 1)
(21, 1)
(21, 1)
2


In [275]:
beta.squeeze()

array([-1.98490652e+00, -3.55981469e-02,  1.31147781e+00,  1.69592916e-01,
       -7.35311432e-02,  3.12657330e-01, -2.36592796e-01, -2.08494417e-01,
       -8.26566822e-02, -2.36311641e-01, -2.92380679e-01,  3.94080487e-02,
        7.18146301e-03,  1.80300116e-02, -2.07373844e-02,  2.19729024e-02,
        1.01442142e-01, -4.39084068e-02,  7.69400674e-03,  1.33923596e-03,
        6.94512382e-04])

In [276]:
log_lmbda_hat.squeeze()

array([-2.02050467, -1.95701843, -1.8950094 , ..., -1.47341303,
       -1.49044594, -1.4640558 ])

In [277]:
GLM.spike_trains_neg_log_likelihood(log_lmbda_hat, model.response[:,np.newaxis]) \
    + L2_pen * np.linalg.norm(beta*penalty_vec)**2

28187.88213652896

# old version

In [23]:
sys.path.append("/home/qix/MultiNeuronGLM/IPRFfunctions/")
import smoothing_spline

In [41]:
spikes = V1.spike_train.iloc[15,0][np.newaxis, :]
spikes = np.vstack((spikes, spikes))
time_line = V1.time_line
spmodel = smoothing_spline.SmoothingSpline()
log_lambda_hat, (beta, beta_baseline, log_lambda_offset, hessian, hessian_baseline, nll) \
    = spmodel.poisson_regression_smoothing_spline(spikes, time_line, num_knots=10)


In [32]:
basis, Omega = spmodel.construct_basis_omega(time_line, knots=10)

In [37]:
GLM.spike_trains_neg_log_likelihood(np.array([1,0,1])[:,np.newaxis],
                                   np.array([0.5,0.3,0.5])[:,np.newaxis], 
                                   trial_wise=True)

array([5.43656366])

In [134]:
aaa = np.array([1,0,1])
bbb = np.array([0.5,0.3,0.5])[:,np.newaxis]
GLM.spike_trains_neg_log_likelihood(aaa,
                                   np.hstack((bbb,bbb)), 
                                   trial_wise=True)

10.87312731383618

True

In [130]:
ccc = np.zeros((10, 1))
ccc[0] = 1
ccc[1:]

array([[0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.]])